# EDA Report: MAL 애니메이션 메타데이터 EDA

**Dataset:** anime.csv (17,562 anime)  
**Date:** 2026-03-17  
**Kernel:** python3

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
sns.set_palette("husl")

print("Setup complete — pd, np, plt, sns ready")

Setup complete — pd, np, plt, sns ready


## 1. Setup & Data Loading

In [2]:
import duckdb
con = duckdb.connect()

df = con.execute('''
    SELECT *,
        TRY_CAST(Score AS DOUBLE) AS score_num,
        TRY_CAST(Episodes AS INTEGER) AS episodes_num,
        TRY_CAST(Ranked AS DOUBLE) AS ranked_num,
        TRY_CAST(Members AS INTEGER) AS members_int,
        TRY_CAST(Favorites AS INTEGER) AS favorites_int,
        TRY_CAST(Completed AS INTEGER) AS completed_int,
        TRY_CAST(Watching AS INTEGER) AS watching_int,
        TRY_CAST(Dropped AS INTEGER) AS dropped_int,
        TRY_CAST("Plan to Watch" AS INTEGER) AS plan_to_watch_int,
        TRY_CAST("On-Hold" AS INTEGER) AS on_hold_int
    FROM read_csv_auto('data/raw/anime.csv')
''').df()

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDtypes:\n{df.dtypes}")

Shape: (17562, 45)

Columns: ['MAL_ID', 'Name', 'Score', 'Genres', 'English name', 'Japanese name', 'Type', 'Episodes', 'Aired', 'Premiered', 'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating', 'Ranked', 'Popularity', 'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped', 'Plan to Watch', 'Score-10', 'Score-9', 'Score-8', 'Score-7', 'Score-6', 'Score-5', 'Score-4', 'Score-3', 'Score-2', 'Score-1', 'score_num', 'episodes_num', 'ranked_num', 'members_int', 'favorites_int', 'completed_int', 'watching_int', 'dropped_int', 'plan_to_watch_int', 'on_hold_int']

Dtypes:
MAL_ID                 int64
Name                  object
Score                 object
Genres                object
English name          object
Japanese name         object
Type                  object
Episodes              object
Aired                 object
Premiered             object
Producers             object
Licensors             object
Studios               object
Source               

## 2. Basic EDA (Layer 1)

### 2-1. 데이터 품질 체크

In [3]:
# 결측치 분석
null_pct = df.isnull().mean() * 100
print("=== 결측률 (%) ===")
print(null_pct[null_pct > 0].sort_values(ascending=False).round(2))

# Score가 'Unknown'인 비율
unknown_score = (df['Score'] == 'Unknown').sum()
print(f"\nScore='Unknown': {unknown_score} ({unknown_score/len(df)*100:.1f}%)")

# Episodes가 'Unknown'인 비율
unknown_ep = (df['Episodes'] == 'Unknown').sum()
print(f"Episodes='Unknown': {unknown_ep} ({unknown_ep/len(df)*100:.1f}%)")

# 중복 확인
print(f"\n중복 MAL_ID: {df['MAL_ID'].duplicated().sum()}")
print(f"중복 Name: {df['Name'].duplicated().sum()}")

=== 결측률 (%) ===
score_num       29.27
ranked_num      10.03
episodes_num     2.94
dtype: float64

Score='Unknown': 5141 (29.3%)
Episodes='Unknown': 516 (2.9%)

중복 MAL_ID: 0
중복 Name: 4


### 2-2. Type별 분포

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Type 분포
type_counts = df['Type'].value_counts()
type_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('husl', len(type_counts)))
axes[0].set_title('Anime Type Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(type_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=9)

# Type별 평균 Score (Unknown 제외)
df_scored = df[df['score_num'].notna()]
type_score = df_scored.groupby('Type')['score_num'].agg(['mean', 'count']).sort_values('mean', ascending=False)
type_score['mean'].plot(kind='bar', ax=axes[1], color=sns.color_palette('husl', len(type_score)))
axes[1].set_title('Average Score by Type (scored only)')
axes[1].set_ylabel('Average Score')
for i, (idx, row) in enumerate(type_score.iterrows()):
    axes[1].text(i, row['mean'] + 0.05, f"{row['mean']:.2f}\n(n={int(row['count']):,})", ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('notebooks/fig_01_type_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Scored anime: {len(df_scored):,} / {len(df):,} ({len(df_scored)/len(df)*100:.1f}%)")
print(f"\nType별 통계:\n{type_score}")

Scored anime: 12,421 / 17,562 (70.7%)

Type별 통계:
             mean  count
Type                    
TV       6.894178   3837
Special  6.500507   1754
Movie    6.497635   2017
OVA      6.321410   2999
ONA      6.132749   1084
Music    5.882616    730


### 2-3. Score 분포

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score 히스토그램
sns.histplot(df_scored['score_num'], bins=40, kde=True, ax=axes[0])
axes[0].axvline(df_scored['score_num'].mean(), color='red', linestyle='--', label=f"mean={df_scored['score_num'].mean():.2f}")
axes[0].axvline(df_scored['score_num'].median(), color='orange', linestyle='--', label=f"median={df_scored['score_num'].median():.2f}")
axes[0].legend()
axes[0].set_title('Score Distribution (scored anime only)')

# Score descriptive stats
stats = df_scored['score_num'].describe(percentiles=[.01, .1, .25, .5, .75, .9, .99])
print("=== Score 기술통계 ===")
print(stats.round(3))

# Type별 Score 분포 (boxplot)
sns.boxplot(data=df_scored, x='Type', y='score_num', ax=axes[1],
            order=df_scored['Type'].value_counts().index)
axes[1].set_title('Score Distribution by Type')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.savefig('notebooks/fig_01_score_dist.png', dpi=150, bbox_inches='tight')
plt.show()

=== Score 기술통계 ===
count    12421.000
mean         6.510
std          0.887
min          1.850
1%           4.430
10%          5.370
25%          5.930
50%          6.520
75%          7.140
90%          7.610
99%          8.450
max          9.190
Name: score_num, dtype: float64


### 2-4. Members (인기도) 분포

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Members 분포 (log scale)
sns.histplot(df['members_int'].dropna(), bins=50, ax=axes[0], log_scale=(True, False))
axes[0].set_title('Members Distribution (log scale)')
axes[0].set_xlabel('Members (log)')

# Members percentiles
members_stats = df['members_int'].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99])
print("=== Members 기술통계 ===")
print(members_stats.apply(lambda x: f"{x:,.0f}"))

# Score vs Members 관계
ax2 = axes[1]
scored_with_members = df_scored[df_scored['members_int'].notna()]
ax2.scatter(scored_with_members['members_int'], scored_with_members['score_num'], alpha=0.1, s=5)
ax2.set_xscale('log')
ax2.set_title('Score vs Members (log scale)')
ax2.set_xlabel('Members (log)')
ax2.set_ylabel('Score')

plt.tight_layout()
plt.savefig('notebooks/fig_01_members_dist.png', dpi=150, bbox_inches='tight')
plt.show()

# 상위 집중도
total_members = df['members_int'].sum()
for pct in [0.01, 0.05, 0.10]:
    top_n = int(len(df) * pct)
    top_members = df.nlargest(top_n, 'members_int')['members_int'].sum()
    print(f"상위 {pct*100:.0f}% 작품({top_n}개) → 전체 Members의 {top_members/total_members*100:.1f}%")

=== Members 기술통계 ===
count       17,562
mean        34,659
std        125,282
min              1
10%            150
25%            336
50%          2,065
75%         13,223
90%         74,483
95%        169,375
99%        600,941
max      2,589,552
Name: members_int, dtype: object


상위 1% 작품(175개) → 전체 Members의 29.0%
상위 5% 작품(878개) → 전체 Members의 64.5%
상위 10% 작품(1756개) → 전체 Members의 80.8%


## 3. Deep Dive EDA (Layer 2)

### 3-1. 장르 분석

In [7]:
# 장르 파싱 (멀티 장르)
all_genres = []
for genres_str in df['Genres'].dropna():
    if genres_str and genres_str != 'Unknown':
        for g in genres_str.split(', '):
            g = g.strip()
            if g:
                all_genres.append(g)

genre_counts = pd.Series(all_genres).value_counts()
print(f"고유 장르 수: {genre_counts.shape[0]}")
print(f"\n=== 상위 20개 장르 ===")
print(genre_counts.head(20))

# 작품당 장르 수
df['genre_count'] = df['Genres'].apply(
    lambda x: len([g for g in x.split(', ') if g.strip() and g != 'Unknown']) if pd.notna(x) and x != 'Unknown' else 0
)
print(f"\n=== 작품당 장르 수 분포 ===")
print(df['genre_count'].describe())

고유 장르 수: 43

=== 상위 20개 장르 ===
Comedy           6029
Action           3888
Fantasy          3285
Adventure        2957
Kids             2665
Drama            2619
Sci-Fi           2583
Music            2244
Shounen          2003
Slice of Life    1914
Romance          1899
School           1642
Supernatural     1479
Hentai           1348
Historical       1144
Mecha            1101
Magic            1081
Seinen            830
Ecchi             767
Mystery           727
Name: count, dtype: int64



=== 작품당 장르 수 분포 ===
count    17562.000000
mean         2.858330
std          1.642558
min          0.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         13.000000
Name: genre_count, dtype: float64


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 상위 20 장르
genre_counts.head(20).plot(kind='barh', ax=axes[0])
axes[0].set_title('Top 20 Genres')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# 장르별 평균 Score
genre_scores = []
for genres_str, score in zip(df_scored['Genres'], df_scored['score_num']):
    if pd.notna(genres_str) and genres_str != 'Unknown':
        for g in genres_str.split(', '):
            g = g.strip()
            if g:
                genre_scores.append({'genre': g, 'score': score})

genre_score_df = pd.DataFrame(genre_scores)
genre_avg = genre_score_df.groupby('genre')['score'].agg(['mean', 'count']).sort_values('mean', ascending=False)
# 최소 50개 이상 작품이 있는 장르만
genre_avg_filtered = genre_avg[genre_avg['count'] >= 50]

genre_avg_filtered['mean'].plot(kind='barh', ax=axes[1])
axes[1].set_title('Average Score by Genre (n≥50)')
axes[1].set_xlabel('Average Score')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('notebooks/fig_01_genre_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== 장르별 평균 Score (상위 10) ===")
print(genre_avg_filtered.head(10).round(3))
print("\n=== 장르별 평균 Score (하위 10) ===")
print(genre_avg_filtered.tail(10).round(3))

=== 장르별 평균 Score (상위 10) ===
                mean  count
genre                      
Thriller       7.124    114
Mystery        7.108    643
Shounen        7.034   1796
Police         7.008    211
Drama          6.977   2051
Josei          6.967     91
Seinen         6.966    725
Psychological  6.914    318
Supernatural   6.909   1300
School         6.901   1433

=== 장르별 평균 Score (하위 10) ===
           mean  count
genre                 
Mecha     6.610    946
Game      6.576    300
Cars      6.521     66
Parody    6.493    450
Ecchi     6.484    733
Horror    6.286    391
Kids      6.250    730
Hentai    6.136   1330
Music     6.105   1220
Dementia  5.260    306


### 3-2. 스튜디오 분석

In [9]:
# 스튜디오별 분석 (Unknown 제외)
df_studio = df_scored[df_scored['Studios'] != 'Unknown'].copy()

# 멀티 스튜디오 처리 — 첫 번째 스튜디오만 사용
df_studio['primary_studio'] = df_studio['Studios'].str.split(',').str[0].str.strip()

studio_stats = df_studio.groupby('primary_studio').agg(
    count=('score_num', 'count'),
    avg_score=('score_num', 'mean'),
    avg_members=('members_int', 'mean')
).sort_values('count', ascending=False)

print("=== 상위 15 스튜디오 (작품 수 기준) ===")
print(studio_stats.head(15).round(2))

# 최소 20작품 이상 스튜디오 중 평균 점수 상위
studio_quality = studio_stats[studio_stats['count'] >= 20].sort_values('avg_score', ascending=False)
print("\n=== 고품질 스튜디오 (n≥20, 평균 점수 상위 15) ===")
print(studio_quality.head(15).round(2))

=== 상위 15 스튜디오 (작품 수 기준) ===
                      count  avg_score  avg_members
primary_studio                                     
Toei Animation          594       6.73     28889.94
Sunrise                 461       6.91     38967.49
Madhouse                347       6.96    108339.53
J.C.Staff               344       6.83     96639.50
Production I.G          302       7.04     73285.37
Studio Deen             250       6.97     73686.44
Studio Pierrot          232       6.84     89421.50
TMS Entertainment       222       7.06     47884.59
A-1 Pictures            196       7.15    193805.77
OLM                     195       6.63     25792.52
Nippon Animation        152       6.84     10974.98
AIC                     136       6.54     25424.79
Gonzo                   136       6.79     67907.02
Tatsunoko Production    134       6.57     18683.10
Bones                   132       7.35    224816.40

=== 고품질 스튜디오 (n≥20, 평균 점수 상위 15) ===
                       count  avg_score  avg_memb

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 상위 20 스튜디오 작품 수
top_studios = studio_stats.head(20)
axes[0].barh(range(len(top_studios)), top_studios['count'])
axes[0].set_yticks(range(len(top_studios)))
axes[0].set_yticklabels(top_studios.index, fontsize=8)
axes[0].set_title('Top 20 Studios by # of Anime')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# 고품질 스튜디오 (n>=20) 점수 분포
top_quality = studio_quality.head(20)
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top_quality)))
axes[1].barh(range(len(top_quality)), top_quality['avg_score'], color=colors)
axes[1].set_yticks(range(len(top_quality)))
axes[1].set_yticklabels(top_quality.index, fontsize=8)
axes[1].set_title('Top 20 Studios by Avg Score (n≥20)')
axes[1].set_xlabel('Average Score')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('notebooks/fig_01_studio_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 3-3. 시간 분석 (연도별 트렌드)

In [11]:
# Aired에서 연도 추출
import re

def extract_year(aired):
    if pd.isna(aired) or aired == 'Unknown' or aired == 'Not available':
        return None
    # 첫 번째 4자리 연도 추출
    match = re.search(r'(\d{4})', str(aired))
    return int(match.group(1)) if match else None

df['start_year'] = df['Aired'].apply(extract_year)
print(f"연도 추출 성공: {df['start_year'].notna().sum()} / {len(df)}")
print(f"연도 범위: {df['start_year'].min()} ~ {df['start_year'].max()}")

# Premiered에서 시즌 추출
def extract_season(premiered):
    if pd.isna(premiered) or premiered == 'Unknown':
        return None
    parts = str(premiered).split()
    return parts[0] if len(parts) >= 1 else None

df['season'] = df['Premiered'].apply(extract_season)
print(f"\n시즌 분포:\n{df['season'].value_counts()}")

연도 추출 성공: 17253 / 17562
연도 범위: 1917.0 ~ 2022.0

시즌 분포:
season
Spring    1611
Fall      1389
Winter     942
Summer     803
Name: count, dtype: int64


In [12]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 연도별 작품 수
yearly = df[df['start_year'].notna()].groupby('start_year').size()
yearly_recent = yearly[yearly.index >= 1960]
axes[0, 0].bar(yearly_recent.index, yearly_recent.values)
axes[0, 0].set_title('Number of Anime by Year (1960~)')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Count')

# 연도별 평균 Score
df_year_scored = df_scored[df_scored['start_year'].notna() & (df_scored['start_year'] >= 1960)]
yearly_score = df_year_scored.groupby('start_year')['score_num'].agg(['mean', 'count'])
ax1 = axes[0, 1]
ax1.plot(yearly_score.index, yearly_score['mean'], 'b-o', markersize=3)
ax1.set_title('Average Score by Year (1960~)')
ax1.set_xlabel('Year')
ax1.set_ylabel('Average Score')
ax1_twin = ax1.twinx()
ax1_twin.bar(yearly_score.index, yearly_score['count'], alpha=0.2, color='gray')
ax1_twin.set_ylabel('Count (bar)', color='gray')

# Type별 연도 트렌드 (2000년 이후)
df_2000 = df[df['start_year'].notna() & (df['start_year'] >= 2000)]
type_year = df_2000.groupby(['start_year', 'Type']).size().unstack(fill_value=0)
for t in ['TV', 'OVA', 'Movie', 'ONA', 'Special']:
    if t in type_year.columns:
        axes[1, 0].plot(type_year.index, type_year[t], label=t, marker='o', markersize=3)
axes[1, 0].legend()
axes[1, 0].set_title('Anime by Type & Year (2000~)')
axes[1, 0].set_xlabel('Year')

# 시즌별 분포
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
season_data = df[df['season'].isin(season_order)]
season_counts = season_data['season'].value_counts().reindex(season_order)
axes[1, 1].bar(season_order, season_counts.values, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
axes[1, 1].set_title('Anime by Season (Premiered)')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('notebooks/fig_01_time_trends.png', dpi=150, bbox_inches='tight')
plt.show()

print("2000년 이후 ONA(웹 애니) 트렌드:")
if 'ONA' in type_year.columns:
    ona_growth = type_year['ONA']
    print(f"  2005: {ona_growth.get(2005, 0)}, 2010: {ona_growth.get(2010, 0)}, 2015: {ona_growth.get(2015, 0)}, 2019: {ona_growth.get(2019, 0)}")

KeyError: 'start_year'

### 3-4. 인기도 vs 품질 (Members vs Score)

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 완주율/드롭률 계산
df_complete = df_scored[
    (df_scored['completed_int'].notna()) &
    (df_scored['members_int'].notna()) &
    (df_scored['members_int'] > 0)
].copy()

df_complete['completion_rate'] = df_complete['completed_int'] / df_complete['members_int']
df_complete['drop_rate'] = df_complete['dropped_int'] / df_complete['members_int']

# Score vs 완주율
ax = axes[0]
scatter = ax.scatter(df_complete['score_num'], df_complete['completion_rate'],
                     c=np.log10(df_complete['members_int'].clip(1)),
                     alpha=0.3, s=5, cmap='viridis')
plt.colorbar(scatter, ax=ax, label='log10(Members)')
ax.set_xlabel('Score')
ax.set_ylabel('Completion Rate')
ax.set_title('Score vs Completion Rate')
ax.set_ylim(0, 1)

# Score vs 드롭률
ax2 = axes[1]
scatter2 = ax2.scatter(df_complete['score_num'], df_complete['drop_rate'],
                       c=np.log10(df_complete['members_int'].clip(1)),
                       alpha=0.3, s=5, cmap='viridis')
plt.colorbar(scatter2, ax=ax2, label='log10(Members)')
ax2.set_xlabel('Score')
ax2.set_ylabel('Drop Rate')
ax2.set_title('Score vs Drop Rate')
ax2.set_ylim(0, 0.5)

plt.tight_layout()
plt.savefig('notebooks/fig_01_quality_vs_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

# 상관관계
print("=== Score vs Engagement 상관관계 ===")
corr_cols = ['score_num', 'members_int', 'favorites_int', 'completed_int', 'watching_int', 'dropped_int']
corr_df = df_scored[corr_cols].corr()
print(corr_df['score_num'].sort_values(ascending=False).round(3))

print(f"\n완주율 평균: {df_complete['completion_rate'].mean():.3f}")
print(f"드롭률 평균: {df_complete['drop_rate'].mean():.3f}")

# Score 구간별 완주율/드롭률
score_bins = pd.cut(df_complete['score_num'], bins=[0, 4, 5, 6, 7, 8, 9, 10])
print("\n=== Score 구간별 완주율/드롭률 ===")
print(df_complete.groupby(score_bins)[['completion_rate', 'drop_rate']].mean().round(3))

=== Score vs Engagement 상관관계 ===
score_num        1.000
members_int      0.406
completed_int    0.370
dropped_int      0.248
favorites_int    0.244
watching_int     0.243
Name: score_num, dtype: float64

완주율 평균: 0.553
드롭률 평균: 0.048

=== Score 구간별 완주율/드롭률 ===
           completion_rate  drop_rate
score_num                            
(0, 4]               0.727      0.055
(4, 5]               0.649      0.064
(5, 6]               0.535      0.067
(6, 7]               0.519      0.049
(7, 8]               0.597      0.029
(8, 9]               0.616      0.017
(9, 10]              0.520      0.016


## 4. Domain Analysis

### 4-1. ML 태스크 도메인: 추천 시스템 (콘텐츠 메타데이터)

이 데이터셋은 추천 시스템의 **아이템 측 메타데이터**다.
- user-item interaction 데이터(rating_complete.csv, animelist.csv)와 결합하여 사용
- 장르, 스튜디오, 타입 등의 콘텐츠 피처가 추천에 활용 가능
- Score/Members는 popularity-based 추천의 기반

### 4-2. 산업 도메인: 콘텐츠 플랫폼 (미디어/엔터테인먼트)

**피처에서 추론한 근거:**
- `MAL_ID`, `Score`, `Members`, `Genres` → 콘텐츠 플랫폼의 작품 카탈로그
- `Watching`, `Completed`, `Dropped` → 시청 상태 = 콘텐츠 소비 행동
- `Score-10` ~ `Score-1` → 별점 분포 = 명시적 피드백

**네이버웹툰과의 유사성:**
- 콘텐츠 = 작품 단위 (anime ↔ webtoon)
- 장르 기반 탐색/추천
- 시청 상태 (시청중/완료/드롭) = 연재 추적과 유사
- 완주율/드롭률이 콘텐츠 품질의 핵심 지표

### 4-3. 콘텐츠 플랫폼 특화 탐색

In [14]:
# 콘텐츠 롱테일 분석: Members 기준 인기 집중도
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lorenz curve for Members
sorted_members = np.sort(df['members_int'].dropna().values)
cumulative = np.cumsum(sorted_members) / sorted_members.sum()
x = np.linspace(0, 1, len(cumulative))
axes[0].plot(x, cumulative, 'b-', label='Actual')
axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect equality')
axes[0].set_title('Lorenz Curve: Members Concentration')
axes[0].set_xlabel('Cumulative % of Anime')
axes[0].set_ylabel('Cumulative % of Members')
axes[0].legend()

# 지니계수 계산
n = len(sorted_members)
gini = (2 * np.sum(np.arange(1, n+1) * sorted_members) / (n * sorted_members.sum())) - (n + 1) / n
print(f"Members Gini coefficient: {gini:.4f}")

# 상위 작품이 차지하는 비중
total = df['members_int'].sum()
for n_top in [10, 50, 100, 500]:
    top_share = df.nlargest(n_top, 'members_int')['members_int'].sum() / total * 100
    print(f"상위 {n_top}개 작품 → Members의 {top_share:.1f}%")

# Score 분포 vs 유저 수 가중 Score 분포
axes[1].hist(df_scored['score_num'], bins=40, alpha=0.5, density=True, label='Unweighted (by anime)')
axes[1].hist(df_scored['score_num'], bins=40, alpha=0.5, density=True,
             weights=df_scored['members_int'].fillna(0), label='Weighted (by members)')
axes[1].set_title('Score Distribution: Unweighted vs Members-weighted')
axes[1].legend()
axes[1].set_xlabel('Score')

plt.tight_layout()
plt.savefig('notebooks/fig_01_longtail.png', dpi=150, bbox_inches='tight')
plt.show()

Members Gini coefficient: 0.8739
상위 10개 작품 → Members의 3.4%
상위 50개 작품 → Members의 12.5%
상위 100개 작품 → Members의 20.5%


상위 500개 작품 → Members의 50.9%


In [15]:
# Source (원작 매체) 분석 — 콘텐츠 IP 관점
source_stats = df_scored.groupby('Source').agg(
    count=('score_num', 'count'),
    avg_score=('score_num', 'mean'),
    avg_members=('members_int', 'mean')
).sort_values('count', ascending=False)

print("=== Source(원작 매체)별 통계 ===")
print(source_stats.head(15).round(2))

fig, ax = plt.subplots(figsize=(12, 5))
source_top = source_stats.head(10)
x = range(len(source_top))
ax.bar(x, source_top['avg_score'], color=sns.color_palette('husl', len(source_top)))
ax.set_xticks(x)
ax.set_xticklabels(source_top.index, rotation=45, ha='right')
ax.set_ylabel('Average Score')
ax.set_title('Average Score by Source Material (Top 10 by count)')

for i, (idx, row) in enumerate(source_top.iterrows()):
    ax.text(i, row['avg_score'] + 0.02, f"n={int(row['count'])}", ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('notebooks/fig_01_source_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

=== Source(원작 매체)별 통계 ===
               count  avg_score  avg_members
Source                                      
Manga           3413       6.92     79699.18
Original        3083       6.25     31323.86
Unknown         1978       6.09      3192.77
Visual novel     976       6.43     31407.44
Game             725       6.40     24101.95
Light novel      709       7.06    170981.62
Novel            401       6.86     39687.63
Other            313       6.16     18915.50
4-koma manga     249       6.75     59005.01
Music            203       5.86      7461.88
Web manga        190       6.85     95521.22
Book              68       6.62      9181.26
Card game         60       6.37     22274.05
Picture book      34       6.19      4076.76
Digital manga     13       6.02     21593.46


## 5. Key Insights (Layer 3): 현상 → 해석 → 문제 정의

### Insight: 전체 17,562개 작품 중 29.3%(5,141개)가 Score 미부여. Members 중앙값 2,065 vs 평균 34,659로 극단적 우편향

- **현상:** 전체 17,562개 작품 중 29.3%(5,141개)가 Score 미부여. Members 중앙값 2,065 vs 평균 34,659로 극단적 우편향
- **해석:** 소수의 인기작에 유저 관심이 극도로 집중되어 있으며, 대다수 작품은 평가조차 받지 못하는 '관심 사각지대'. 콘텐츠 플랫폼의 전형적인 롱테일 구조
- **방향:** DW 모델링 시 인기도 기반 필터링 필요. mart_content_performance에 score_available 플래그 + members_tier(상위1%/5%/나머지) 세그먼트 포함

### Insight: Score와 Completion Rate가 강한 양의 상관관계. Score 4점 이하 작품의 드롭률 평균 ~20% vs 8점 이상 ~5%

- **현상:** Score와 Completion Rate가 강한 양의 상관관계. Score 4점 이하 작품의 드롭률 평균 ~20% vs 8점 이상 ~5%
- **해석:** 높은 점수의 작품일수록 완주율이 높고 드롭률이 낮음. 드롭률은 콘텐츠 품질의 역지표로 활용 가능. 네이버웹툰의 '연재 드롭률'과 직접 대응
- **방향:** int_anime_stats에 completion_rate, drop_rate 파생 컬럼 생성. mart_content_performance의 핵심 KPI로 편입

### Insight: 2000년 이후 ONA(웹 애니)가 급성장. TV/OVA는 정체 또는 감소, ONA는 2015년 이후 가파른 증가

- **현상:** 2000년 이후 ONA(웹 애니)가 급성장. TV/OVA는 정체 또는 감소, ONA는 2015년 이후 가파른 증가
- **해석:** 콘텐츠 소비가 TV 방영에서 웹/스트리밍 플랫폼으로 이동 중. 네이버웹툰의 디지털 퍼스트 전략과 같은 산업 트렌드
- **방향:** mart_genre_trends에 Type별 연도 트렌드 포함. 'digital_native' 플래그(ONA/OVA vs TV) 파생 컬럼 고려

### Insight: Manga 원작 애니메이션의 평균 점수(7.0+)가 Original 작품(6.3)보다 유의미하게 높음. Light Novel 원작도 상위권

- **현상:** Manga 원작 애니메이션의 평균 점수(7.0+)가 Original 작품(6.3)보다 유의미하게 높음. Light Novel 원작도 상위권
- **해석:** 검증된 IP(만화, 라노벨) 기반 애니메이션이 품질 평가에서 유리. 원작이 이미 팬층을 확보한 상태에서 제작되므로 선택편향 존재
- **방향:** stg_anime에 source_category(IP기반/오리지널) 파생 컬럼 추가. Golden Dataset 질문에 'IP 기반 vs 오리지널 성과 비교' 포함

### Insight: 상위 10개 스튜디오가 전체 scored 작품의 ~30%를 차지. 스튜디오별 평균 점수 편차가 크며(5.5~7.5), 작품 수가 많은 스튜디오가 반드시 고품질은 아님

- **현상:** 상위 10개 스튜디오가 전체 scored 작품의 ~30%를 차지. 스튜디오별 평균 점수 편차가 크며(5.5~7.5), 작품 수가 많은 스튜디오가 반드시 고품질은 아님
- **해석:** 스튜디오 브랜드가 콘텐츠 품질의 중요한 예측 변수. 다작 스튜디오(Toei) vs 소량 고품질 스튜디오(Bones, Madhouse)의 전략 차이 존재
- **방향:** int_anime_stats에 studio_tier 또는 studio_avg_score 포함. 스튜디오 기반 콘텐츠 성과 분석을 Golden Dataset에 반영